# Modelización mediante regresiones lineales múltiples por ciudad

## Objetivo

Este notebook desarrolla la fase de **modelización e interpretación** del TFM.

La finalidad principal consiste en responder a la siguiente cuestión:

> **¿Qué características de los pisos se asocian con mayor intensidad al precio de oferta dentro de cada ciudad y cómo varía esa relación entre mercados urbanos?**

Desde el diseño metodológico se estiman dos especificaciones para cada ciudad:

- **Modelo base:** incorpora las características fundamentales del piso.
- **Modelo ampliado:** añade, además, información de planta, estado de conservación, distancia al punto central de referencia, no linealidad de la superficie, aire acondicionado y trastero.

El modelo base funciona como referencia interna para comprobar si la incorporación de las variables adicionales mejora el ajuste. No constituye una versión anterior del trabajo, sino una parte del propio diseño del análisis.

Las variables explicativas y `log_price` se estandarizan mediante **Z-score dentro de cada ciudad**, lo que permite comparar la magnitud de los coeficientes beta. También se revisan R² ajustado, AIC/BIC, VIF, significación estadística mediante errores estándar robustos HC3 y diagnósticos de residuos.

El análisis es **explicativo y asociativo, no causal**.


## 1. Importación de librerías y configuración

La estandarización Z-score se calcula directamente con `pandas` y `numpy`, de modo que no es necesario instalar `scikit-learn`. `statsmodels` se emplea para estimar las regresiones y realizar los diagnósticos.

In [ ]:
# =====================================================
# IMPORTACIÓN DE LIBRERÍAS
# =====================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

### Interpretación

En el proyecto se utilizan únicamente librerías ya habituales. El Z-score se implementará mediante la fórmula `(x - media) / desviación estándar`, con desviación estándar calculada con `ddof=0`, equivalente a la estandarización empleada de forma habitual para obtener coeficientes beta comparables.

## 2. Carga y definición de la muestra

### 2.1 Carga del conjunto procesado

Se carga el mismo `viviendas_limpio.csv` utilizado en el EDA. Las transformaciones específicas del modelo se aplican sobre una copia y no alteran el archivo original.

In [ ]:
# =====================================================
# RUTAS Y CARGA DE DATOS
# =====================================================

# =====================================================
# DETECCIÓN DE LA RAÍZ DEL PROYECTO
# =====================================================

# Estructura esperada:
# TFM-Analisis-Inmobiliario/
# ├── data/raw
# ├── data/processed
# ├── notebooks
# └── reports/{figures,tables}
#
# La detección funciona tanto si VS Code ejecuta el notebook
# desde la raíz del proyecto como desde la carpeta notebooks/.
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        ruta for ruta in [cwd, *cwd.parents]
        if (ruta / "data" / "raw").is_dir()
        and (ruta / "notebooks").is_dir()
        and (ruta / "reports").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Se esperaba encontrar las carpetas data/raw, notebooks y reports "
        "en el directorio actual o en alguno de sus directorios superiores."
    )

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "viviendas_limpio.csv"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "No se encuentra viviendas_limpio.csv. "
        f"Ejecuta primero el Notebook 02. Ruta esperada: {DATA_PATH}"
    )

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Dataset utilizado: {DATA_PATH}")
print(f"Tablas: {TABLES_DIR}")
print(f"Figuras: {FIGURES_DIR}")

df_viviendas = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Registros cargados: {len(df_viviendas):,}")
print(f"Variables disponibles: {df_viviendas.shape[1]}")


### Interpretación

El punto de partida es el conjunto procesado de **54.004 anuncios y 62 variables**. El dataset se conserva intacto y se crea una muestra específica para la regresión.

### 2.2 Comprobación de variables necesarias

Antes del ajuste se verifica que todas las columnas requeridas estén disponibles. Esta validación impide continuar con un modelo incompleto si cambia la estructura del CSV.

In [ ]:
# =====================================================
# COMPROBACIÓN DE VARIABLES
# =====================================================

columnas_necesarias = [
    "price", "size", "rooms", "bathrooms",
    "ciudad", "propertyType",
    "hasLift", "exterior", "floor", "status",
    "newDevelopment", "latitude", "longitude",
    "features/hasTerrace",
    "features/hasSwimmingPool",
    "features/hasGarden",
    "features/hasAirConditioning",
    "features/hasBoxRoom",
    "parkingSpace/hasParkingSpace",
]

faltantes = [
    c for c in columnas_necesarias
    if c not in df_viviendas.columns
]

if faltantes:
    raise KeyError(
        "Faltan columnas necesarias: " + ", ".join(faltantes)
    )

print("Comprobación superada: están disponibles todas las variables necesarias.")

### Interpretación

La comprobación confirma que el conjunto procesado contiene todas las variables necesarias para construir las dos especificaciones del modelo.

### 2.3 Selección de pisos y tratamiento de valores extremos

Se conservan únicamente los registros `propertyType == "flat"` para trabajar con una tipología homogénea.

Después se eliminan observaciones sin información en las variables esenciales y, **dentro de cada ciudad**, se excluyen los valores situados por debajo del percentil 1 o por encima del percentil 99 de `price` y `size`. El criterio coincide con el utilizado en el EDA comparativo.

In [ ]:
# =====================================================
# MUESTRA DE MODELIZACIÓN
# =====================================================

df_modelo = df_viviendas[
    df_viviendas["propertyType"]
    .astype(str)
    .str.lower()
    .eq("flat")
].copy()

variables_clave = [
    "price", "size", "rooms", "bathrooms",
    "ciudad", "latitude", "longitude",
]

df_modelo = (
    df_modelo
    .dropna(subset=variables_clave)
    .loc[lambda x: (x["price"] > 0) & (x["size"] > 0)]
    .copy()
)

partes = []
resumen_muestra = []

for ciudad, grupo in df_modelo.groupby("ciudad", sort=True):
    p_price = grupo["price"].quantile([0.01, 0.99])
    p_size = grupo["size"].quantile([0.01, 0.99])

    mascara = (
        grupo["price"].between(
            p_price.loc[0.01],
            p_price.loc[0.99],
        )
        & grupo["size"].between(
            p_size.loc[0.01],
            p_size.loc[0.99],
        )
    )

    grupo_filtrado = grupo.loc[mascara].copy()
    partes.append(grupo_filtrado)

    resumen_muestra.append({
        "Ciudad": ciudad,
        "N inicial": len(grupo),
        "N final": len(grupo_filtrado),
        "Retención (%)": (
            100 * len(grupo_filtrado) / len(grupo)
        ),
    })

df_modelo = pd.concat(partes, ignore_index=True)
resumen_muestra = pd.DataFrame(resumen_muestra)

display(resumen_muestra.round(2))
print(f"Total de pisos modelizados: {len(df_modelo):,}")

### Interpretación

La muestra pasa de **43.891 pisos** a **42.441 observaciones** tras eliminar las observaciones sin información esencial y aplicar el tratamiento de valores extremos. La retención se sitúa aproximadamente entre el 95 % y el 97 % según la ciudad.

Ávila conserva **198 pisos**. El modelo puede estimarse con esta muestra, pero el número de observaciones es claramente menor que en el resto de ciudades y, por tanto, sus resultados deben interpretarse con especial prudencia.


## 3. Preparación de variables para la regresión

### 3.1 Variable objetivo

Se utiliza el logaritmo natural del precio, `log_price = ln(price)`. Esta transformación reduce la asimetría observada en el EDA.

In [ ]:
# =====================================================
# VARIABLE OBJETIVO
# =====================================================

df_modelo["log_price"] = np.log(df_modelo["price"])

display(
    df_modelo[["price", "log_price"]]
    .describe()
    .round(3)
)

### Interpretación

El modelo no intenta explicar directamente el precio bruto, sino `log_price`. Posteriormente esta variable también se estandariza dentro de cada ciudad para obtener coeficientes beta estandarizados.

### 3.2 Variables binarias y valores ausentes

Las variables binarias pueden contener `True`, `False` y `NaN`.

- `True` y `False` se convierten en 1 y 0.
- Los valores ausentes de ascensor, exterior y de las variables del bloque `features/` se imputan con la **mediana de la variable dentro de su ciudad** para conservar la muestra.
- Se generan indicadores de ausencia para revisar el patrón de nulos, aunque esos indicadores de equipamientos no forman parte del modelo final.
- El aparcamiento se trata por separado: en este dataset el bloque `parkingSpace` aparece cuando el anuncio incorpora información de aparcamiento y `hasParkingSpace` no presenta valores `False`, por lo que la ausencia del bloque se codifica como 0 en la variable derivada `hasParking`.
- `newDevelopment` se transforma igualmente a formato binario y sus valores no informados se codifican como 0 en la copia utilizada para la modelización.

Estas transformaciones se realizan únicamente en la copia de modelización; el CSV procesado conserva los valores originales.

In [ ]:
# =====================================================
# VARIABLES BINARIAS
# =====================================================

def convertir_binaria(serie):
    mapa = {
        True: 1, False: 0,
        "True": 1, "False": 0,
        "true": 1, "false": 0,
        "1": 1, "0": 0,
        1: 1, 0: 0,
    }
    return serie.map(mapa).astype("float64")


def crear_binaria_modelo(df, origen, destino):
    valores = convertir_binaria(df[origen])

    df[f"{destino}_missing"] = (
        valores.isna().astype(int)
    )

    mediana_ciudad = (
        valores
        .groupby(df["ciudad"])
        .transform("median")
    )

    df[destino] = (
        valores
        .fillna(mediana_ciudad)
        .fillna(0)
        .astype(float)
    )

    return df


mapeo_binarias = {
    "hasLift": "hasLift",
    "exterior": "exterior",
    "features/hasTerrace": "hasTerrace",
    "features/hasSwimmingPool": "hasSwimmingPool",
    "features/hasGarden": "hasGarden",
    "features/hasAirConditioning": "hasAirConditioning",
    "features/hasBoxRoom": "hasBoxRoom",
}

for origen, destino in mapeo_binarias.items():
    df_modelo = crear_binaria_modelo(
        df_modelo, origen, destino
    )

df_modelo["hasParking"] = (
    convertir_binaria(
        df_modelo["parkingSpace/hasParkingSpace"]
    )
    .fillna(0)
    .astype(float)
)

df_modelo["newDevelopment"] = (
    convertir_binaria(df_modelo["newDevelopment"])
    .fillna(0)
    .astype(float)
)

resumen_binarias = (
    df_modelo[
        [
            "hasLift", "hasTerrace",
            "hasSwimmingPool", "hasGarden",
            "hasParking", "newDevelopment",
            "exterior", "hasAirConditioning",
            "hasBoxRoom",
        ]
    ]
    .describe()
    .T[["mean", "std", "min", "max"]]
)

display(resumen_binarias.round(3))

### Interpretación

Las variables quedan preparadas en una escala numérica compatible con OLS. La imputación se emplea para evitar perder observaciones, pero no debe interpretarse como información observada: constituye una decisión técnica y se considera una limitación del estudio.

### 3.3 Estado de conservación

A partir de `status` se deriva `status_renew`, que toma el valor 1 cuando el anuncio indica que la vivienda necesita reforma. También se conserva `status_missing` para identificar los pocos casos en los que el estado no está informado.

In [ ]:
# =====================================================
# ESTADO DE CONSERVACIÓN
# =====================================================

status_normalizado = (
    df_modelo["status"]
    .astype("string")
    .str.lower()
    .str.strip()
)

df_modelo["status_renew"] = (
    status_normalizado
    .eq("renew")
    .fillna(False)
    .astype(int)
)

df_modelo["status_missing"] = (
    df_modelo["status"]
    .isna()
    .astype(int)
)

display(
    pd.crosstab(
        df_modelo["ciudad"],
        status_normalizado,
        margins=True,
    )
)

### Interpretación

La categoría `renew` permite analizar si la necesidad de reforma se asocia con el precio una vez controladas las demás características. `newdevelopment` no se convierte en otra dummy porque la información de obra nueva ya queda recogida por `newDevelopment`.

### 3.4 Conversión de la planta

`floor` combina números y códigos. Se transforma en una variable numérica aproximada:

- `bj` → 0;
- `en` → 0,5;
- `ss` → -1;
- `st`/`sm` → -0,5;
- `pr` → 1.

Los valores no interpretables se imputan con la mediana de planta de su ciudad. `floor_missing` indica que la planta original no pudo utilizarse.

In [ ]:
# =====================================================
# CONVERSIÓN DE FLOOR
# =====================================================

def convertir_planta(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().lower()

    equivalencias = {
        "bj": 0.0,
        "en": 0.5,
        "ss": -1.0,
        "st": -0.5,
        "sm": -0.5,
        "pr": 1.0,
    }

    if texto in equivalencias:
        return equivalencias[texto]

    try:
        return float(texto.replace(",", "."))
    except ValueError:
        return np.nan


df_modelo["floor_num_raw"] = (
    df_modelo["floor"]
    .apply(convertir_planta)
)

df_modelo["floor_missing"] = (
    df_modelo["floor_num_raw"]
    .isna()
    .astype(int)
)

mediana_floor_ciudad = (
    df_modelo
    .groupby("ciudad")["floor_num_raw"]
    .transform("median")
)

df_modelo["floor_num"] = (
    df_modelo["floor_num_raw"]
    .fillna(mediana_floor_ciudad)
    .fillna(0)
)

display(
    df_modelo
    .groupby("ciudad")["floor_num"]
    .agg(["count", "median", "min", "max"])
    .round(2)
)

### Interpretación

La transformación permite emplear la planta como variable numérica sin eliminar anuncios con formatos de planta especiales. Debe entenderse como una aproximación operativa, no como una escala física perfecta entre todos los códigos originales.

### 3.5 Distancia al punto central de referencia

La localización se resume mediante la distancia en kilómetros entre cada anuncio y un punto central de referencia definido para su ciudad. Se aplica la fórmula de Haversine sobre latitud y longitud.

Esta variable representa **centralidad aproximada**. No sustituye a variables de barrio, calidad urbana, costa, servicios o accesibilidad.

In [ ]:
# =====================================================
# DISTANCIA AL CENTRO
# =====================================================

CENTROS_CIUDAD = {
    "A Coruña": (43.3623, -8.4115),
    "Alicante": (38.3452, -0.4810),
    "Avila": (40.6566, -4.6813),
    "Barcelona": (41.3874, 2.1686),
    "Las Palmas": (28.1235, -15.4363),
    "Madrid": (40.4168, -3.7038),
    "Malaga": (36.7213, -4.4214),
    "Sevilla": (37.3891, -5.9845),
    "Valencia": (39.4699, -0.3763),
    "Zaragoza": (41.6488, -0.8891),
}


def haversine_km(lat, lon, lat0, lon0):
    radio_tierra = 6371.0088

    lat = np.radians(lat.astype(float))
    lon = np.radians(lon.astype(float))
    lat0 = np.radians(float(lat0))
    lon0 = np.radians(float(lon0))

    dlat = lat - lat0
    dlon = lon - lon0

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat0)
        * np.cos(lat)
        * np.sin(dlon / 2) ** 2
    )

    return (
        2
        * radio_tierra
        * np.arcsin(np.sqrt(a))
    )


df_modelo["dist_center_km"] = np.nan

for ciudad, idx in df_modelo.groupby("ciudad").groups.items():
    if ciudad not in CENTROS_CIUDAD:
        raise KeyError(
            f"No se ha definido un centro para {ciudad}."
        )

    lat0, lon0 = CENTROS_CIUDAD[ciudad]

    df_modelo.loc[idx, "dist_center_km"] = haversine_km(
        df_modelo.loc[idx, "latitude"],
        df_modelo.loc[idx, "longitude"],
        lat0,
        lon0,
    )

resumen_distancias = (
    df_modelo
    .groupby("ciudad")["dist_center_km"]
    .describe(percentiles=[0.5, 0.9, 0.99])
    [["count", "mean", "50%", "90%", "99%", "max"]]
    .round(2)
)

display(resumen_distancias)

### Interpretación

La variable ofrece una medida sencilla y reproducible de la distancia al punto central de referencia de cada mercado urbano. La lectura posterior debe restringirse a la centralidad: una mayor distancia no implica necesariamente peor localización, sobre todo en ciudades policéntricas o con zonas costeras de alto valor.

## 4. Especificación y ajuste de los modelos

### 4.1 Variables de los modelos

El **modelo base** incorpora las características esenciales del piso. El **modelo ampliado** suma variables capaces de captar localización, estado y relaciones no lineales.

`size_sq` se obtiene como el cuadrado de la superficie previamente centrada dentro de cada ciudad. Centrar `size` antes de elevarla al cuadrado reduce la colinealidad puramente matemática entre el término lineal y el cuadrático.

In [ ]:
# =====================================================
# VARIABLES DE LOS MODELOS
# =====================================================

VARIABLES_BASE = [
    "size",
    "rooms",
    "bathrooms",
    "hasLift",
    "hasTerrace",
    "hasSwimmingPool",
    "hasGarden",
    "hasParking",
    "newDevelopment",
    "exterior",
]

VARIABLES_AMPLIADAS = VARIABLES_BASE + [
    "size_sq",
    "floor_num",
    "status_renew",
    "dist_center_km",
    "hasAirConditioning",
    "hasBoxRoom",
    "floor_missing",
    "status_missing",
]

VARIABLES_SUSTANTIVAS = [
    v for v in VARIABLES_AMPLIADAS
    if not v.endswith("_missing")
]

print(f"Variables del modelo base: {len(VARIABLES_BASE)}")
print(f"Variables del modelo ampliado: {len(VARIABLES_AMPLIADAS)}")

### Interpretación

La especificación base contempla **10 predictores** y la ampliada **hasta 18**; las variables sin varianza se excluyen automáticamente dentro de cada ciudad. Los indicadores `floor_missing` y `status_missing` funcionan como controles técnicos; no se emplearán para determinar qué rasgo inmobiliario presenta mayor importancia relativa.

### 4.2 Estandarización Z-score y función de ajuste

Para una variable \(x\), el Z-score se calcula como:

z=(x−μ)/σ

La estandarización se aplica **por ciudad** tanto para los predictores como para `log_price`. Así, las betas expresan cambios en desviaciones estándar y pueden compararse entre variables.

La función también:

1. crea `size_sq`;
2. elimina variables sin varianza dentro de una ciudad;
3. estima OLS;
4. calcula errores estándar robustos **HC3**;
5. calcula VIF;
6. almacena R², R² ajustado, AIC/BIC y diagnósticos.

In [ ]:
# =====================================================
# FUNCIONES DE ESTANDARIZACIÓN Y AJUSTE
# =====================================================

def estandarizar_zscore(df):
    desviacion = df.std(ddof=0)

    if (desviacion == 0).any():
        raise ValueError(
            "Se ha intentado estandarizar una variable "
            "sin variación."
        )

    return (
        df - df.mean()
    ) / desviacion


def ajustar_modelo_ciudad(
    df_ciudad,
    ciudad,
    variables,
    nombre_modelo,
):
    datos = df_ciudad.copy()

    # Término cuadrático centrado
    datos["size_centered"] = (
        datos["size"] - datos["size"].mean()
    )
    datos["size_sq"] = (
        datos["size_centered"] ** 2
    )

    variables_presentes = [
        v for v in variables
        if v in datos.columns
    ]

    columnas_uso = (
        variables_presentes + ["log_price"]
    )

    datos[columnas_uso] = (
        datos[columnas_uso]
        .replace([np.inf, -np.inf], np.nan)
    )

    datos = (
        datos
        .dropna(subset=columnas_uso)
        .copy()
    )

    variables_constantes = [
        v for v in variables_presentes
        if datos[v].nunique(dropna=True) <= 1
    ]

    variables_utilizadas = [
        v for v in variables_presentes
        if v not in variables_constantes
    ]

    X = datos[
        variables_utilizadas
    ].astype(float)

    y = datos[
        ["log_price"]
    ].astype(float)

    X_z = estandarizar_zscore(X)

    y_z = (
        estandarizar_zscore(y)
        ["log_price"]
        .rename("z_log_price")
    )

    X_ols = sm.add_constant(
        X_z,
        has_constant="add",
    )

    modelo_ols = sm.OLS(
        y_z,
        X_ols,
    ).fit()

    robusto = (
        modelo_ols
        .get_robustcov_results(cov_type="HC3")
    )

    indices = X_ols.columns

    params = pd.Series(
        robusto.params,
        index=indices,
    )
    bse = pd.Series(
        robusto.bse,
        index=indices,
    )
    tvalues = pd.Series(
        robusto.tvalues,
        index=indices,
    )
    pvalues = pd.Series(
        robusto.pvalues,
        index=indices,
    )
    intervalos = pd.DataFrame(
        robusto.conf_int(),
        index=indices,
    )

    coeficientes = pd.DataFrame({
        "Ciudad": ciudad,
        "Modelo": nombre_modelo,
        "Variable": variables_utilizadas,
        "Beta_estandarizada": (
            params[variables_utilizadas].values
        ),
        "Error_estandar_HC3": (
            bse[variables_utilizadas].values
        ),
        "t_HC3": (
            tvalues[variables_utilizadas].values
        ),
        "p_value_HC3": (
            pvalues[variables_utilizadas].values
        ),
        "IC95_inf_HC3": (
            intervalos.loc[
                variables_utilizadas, 0
            ].values
        ),
        "IC95_sup_HC3": (
            intervalos.loc[
                variables_utilizadas, 1
            ].values
        ),
    })

    coeficientes["Significativa_5pct_HC3"] = (
        coeficientes["p_value_HC3"] < 0.05
    )
    coeficientes["Beta_abs"] = (
        coeficientes["Beta_estandarizada"].abs()
    )

    vif = pd.DataFrame({
        "Ciudad": ciudad,
        "Modelo": nombre_modelo,
        "Variable": variables_utilizadas,
        "VIF": [
            variance_inflation_factor(
                X_z.to_numpy(), i
            )
            for i in range(
                len(variables_utilizadas)
            )
        ],
    })

    residuos = modelo_ols.resid

    jb_stat, jb_p, skew, kurtosis = (
        jarque_bera(residuos)
    )

    bp_stat, bp_p, _, _ = (
        het_breuschpagan(
            residuos,
            X_ols,
        )
    )

    resumen = {
        "Ciudad": ciudad,
        "Modelo": nombre_modelo,
        "N": int(modelo_ols.nobs),
        "N_variables": len(
            variables_utilizadas
        ),
        "R2": modelo_ols.rsquared,
        "R2_ajustado": (
            modelo_ols.rsquared_adj
        ),
        "AIC": modelo_ols.aic,
        "BIC": modelo_ols.bic,
        "VIF_max": vif["VIF"].max(),
        "Breusch_Pagan_p": bp_p,
        "Jarque_Bera_p": jb_p,
        "Variables_excluidas_varianza_cero": (
            ", ".join(variables_constantes)
        ),
    }

    return (
        modelo_ols,
        coeficientes,
        vif,
        resumen,
    )

### Interpretación

El código estima cada ciudad por separado. HC3 no modifica las betas ni el R²; ajusta los errores estándar, p-valores e intervalos de confianza para que la inferencia sea menos sensible a la heterocedasticidad.

Una beta de `0,40`, por ejemplo, indica que un incremento de una desviación estándar en el predictor se asocia con un aumento aproximado de 0,40 desviaciones estándar en `log_price`, manteniendo constantes las demás variables.

Esta celda define la función que se utilizará después para estimar las regresiones de cada ciudad. Al limitarse a la definición de una función, todavía no produce tablas ni gráficos. Los resultados aparecerán en el apartado siguiente, cuando la función se aplique a cada uno de los mercados urbanos.

## 5. Resultados de los modelos

### 5.1 Ajuste por ciudad

Se estiman dos modelos para cada una de las diez ciudades, con un total de veinte regresiones.

In [ ]:
# =====================================================
# AJUSTE DE MODELOS
# =====================================================

modelos = {}
lista_coeficientes = []
lista_vif = []
lista_resumen = []

for ciudad, df_ciudad in df_modelo.groupby(
    "ciudad",
    sort=True,
):
    modelos[ciudad] = {}

    especificaciones = {
        "Base": VARIABLES_BASE,
        "Ampliado": VARIABLES_AMPLIADAS,
    }

    for nombre_modelo, variables in (
        especificaciones.items()
    ):
        (
            modelo,
            coef,
            vif,
            resumen,
        ) = ajustar_modelo_ciudad(
            df_ciudad=df_ciudad,
            ciudad=ciudad,
            variables=variables,
            nombre_modelo=nombre_modelo,
        )

        modelos[ciudad][
            nombre_modelo
        ] = modelo

        lista_coeficientes.append(coef)
        lista_vif.append(vif)
        lista_resumen.append(resumen)

resultados_modelos = pd.DataFrame(
    lista_resumen
)

coeficientes = pd.concat(
    lista_coeficientes,
    ignore_index=True,
)

vif_todos = pd.concat(
    lista_vif,
    ignore_index=True,
)

print(
    f"Modelos ajustados: "
    f"{len(resultados_modelos)}"
)

display(
    resultados_modelos.round(4)
)

### Interpretación

Se ajustan correctamente **20 modelos**. En Ávila, `newDevelopment` se excluye automáticamente por no presentar variación; en el modelo ampliado también se excluye `status_missing` por el mismo motivo.

Las demás ciudades conservan todos los predictores previstos.

### 5.2 Comparación del ajuste

La comparación principal se basa en el **R² ajustado**, ya que penaliza en parte la incorporación de variables adicionales. AIC y BIC solo se comparan **dentro de una misma ciudad y sobre la misma muestra**; sus valores absolutos no sirven para ordenar ciudades.

In [ ]:
# =====================================================
# COMPARACIÓN DE MODELOS
# =====================================================

comparacion = (
    resultados_modelos
    .pivot(
        index="Ciudad",
        columns="Modelo",
        values=[
            "R2",
            "R2_ajustado",
            "AIC",
            "BIC",
            "N",
            "VIF_max",
        ],
    )
)

comparacion.columns = [
    f"{metrica}_{modelo}"
    for metrica, modelo
    in comparacion.columns
]

comparacion = (
    comparacion
    .reset_index()
)

comparacion["Mejora_R2_ajustado"] = (
    comparacion["R2_ajustado_Ampliado"]
    - comparacion["R2_ajustado_Base"]
)

comparacion["Delta_AIC"] = (
    comparacion["AIC_Ampliado"]
    - comparacion["AIC_Base"]
)

comparacion["Delta_BIC"] = (
    comparacion["BIC_Ampliado"]
    - comparacion["BIC_Base"]
)

comparacion = comparacion.sort_values(
    "Mejora_R2_ajustado",
    ascending=False,
)

columnas_mostrar = [
    "Ciudad",
    "R2_ajustado_Base",
    "R2_ajustado_Ampliado",
    "Mejora_R2_ajustado",
    "Delta_AIC",
    "Delta_BIC",
]

display(
    comparacion[columnas_mostrar]
    .round(3)
)
# Figura 5-3: comparación completa de ajuste y criterios de información
tabla_fig_5_3 = comparacion[columnas_mostrar].copy()
tabla_fig_5_3 = tabla_fig_5_3.rename(
    columns={
        "R2_ajustado_Base": "R² aj. base",
        "R2_ajustado_Ampliado": "R² aj. ampliado",
        "Mejora_R2_ajustado": "Mejora R² aj.",
        "Delta_AIC": "ΔAIC",
        "Delta_BIC": "ΔBIC",
    }
)

for columna in tabla_fig_5_3.columns[1:]:
    tabla_fig_5_3[columna] = tabla_fig_5_3[columna].map(
        lambda x: f"{x:.3f}"
    )

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.axis("off")
tabla = ax.table(
    cellText=tabla_fig_5_3.values,
    colLabels=tabla_fig_5_3.columns,
    cellLoc="center",
    loc="center",
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(8)
tabla.scale(1, 1.35)
ax.set_title(
    "Comparación del R² ajustado y de los criterios AIC/BIC",
    pad=16,
)
plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "figura_5_03_comparacion_r2_aic_bic.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


### Interpretación

El modelo ampliado mejora el **R² ajustado en las diez ciudades**.

Las mejoras más elevadas aparecen en:

- **Sevilla:** 0,552 → **0,723** (+0,170).
- **Las Palmas:** 0,478 → **0,644** (+0,167).
- **Madrid:** 0,748 → **0,832** (+0,084).
- **Barcelona:** 0,733 → **0,813** (+0,081).
- **Valencia:** 0,659 → **0,725** (+0,066).

AIC disminuye en las diez ciudades. BIC también mejora en todas salvo Ávila, donde el incremento de complejidad asociado a las variables añadidas no queda compensado por la mejora del ajuste según este criterio.

En conjunto, el **modelo ampliado se adopta como especificación principal** para la interpretación de los resultados.

### 5.3 Coeficientes beta estandarizados

Se seleccionan los coeficientes del modelo ampliado. El signo señala la dirección de la asociación y el valor absoluto de beta permite comparar la intensidad.

In [ ]:
# =====================================================
# COEFICIENTES DEL MODELO AMPLIADO
# =====================================================

coef_ampliado = (
    coeficientes[
        coeficientes["Modelo"].eq(
            "Ampliado"
        )
    ]
    .copy()
)

display(
    coef_ampliado
    .sort_values(
        ["Ciudad", "Beta_abs"],
        ascending=[True, False],
    )
    [
        [
            "Ciudad",
            "Variable",
            "Beta_estandarizada",
            "p_value_HC3",
            "Significativa_5pct_HC3",
        ]
    ]
    .round(4)
)

### Interpretación

La **superficie (`size`) presenta la mayor beta estandarizada en valor absoluto en las diez ciudades**, por lo que es la variable con mayor importancia relativa dentro de los predictores incluidos en el modelo.

Para interpretar las diferencias entre mercados conviene revisar también los factores dominantes cuando se excluyen las variables relacionadas con la superficie.

### 5.4 Variable más influyente excluyendo superficie

Se excluyen `size` y `size_sq` de la clasificación para observar qué característica secundaria presenta la beta absoluta más alta en cada ciudad.

In [ ]:
# =====================================================
# VARIABLE MÁS INFLUYENTE SIN SUPERFICIE
# =====================================================

coef_sustantivos = coef_ampliado[
    ~coef_ampliado["Variable"]
    .str.endswith("_missing")
].copy()

sin_superficie = coef_sustantivos[
    ~coef_sustantivos["Variable"]
    .isin(["size", "size_sq"])
].copy()

mas_influyente_sin_size = (
    sin_superficie
    .sort_values(
        "Beta_abs",
        ascending=False,
    )
    .groupby(
        "Ciudad",
        as_index=False,
    )
    .first()
    [
        [
            "Ciudad",
            "Variable",
            "Beta_estandarizada",
            "p_value_HC3",
            "Beta_abs",
        ]
    ]
)

display(
    mas_influyente_sin_size.round(4)
)
# Figura 5-6: variable secundaria con mayor beta absoluta por ciudad
plot_influyente = mas_influyente_sin_size.sort_values(
    "Beta_abs",
    ascending=True,
)

fig, ax = plt.subplots(figsize=(11, 6))
barras = ax.barh(
    plot_influyente["Ciudad"],
    plot_influyente["Beta_abs"],
)

for barra, (_, fila) in zip(barras, plot_influyente.iterrows()):
    ax.text(
        barra.get_width() + 0.006,
        barra.get_y() + barra.get_height() / 2,
        f'{fila["Variable"]} ({fila["Beta_estandarizada"]:+.3f})',
        va="center",
        fontsize=8,
    )

ax.set_title(
    "Variable con mayor beta absoluta por ciudad, excluyendo superficie"
)
ax.set_xlabel("|Beta estandarizada|")
ax.set_ylabel("Ciudad")
plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "figura_5_06_factor_secundario_dominante.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


### Interpretación

Excluyendo la superficie:

- **Distancia al punto central de referencia** es el factor dominante en Barcelona, Las Palmas, Madrid, Sevilla y Valencia, siempre con beta negativa.
- **Número de baños** domina en A Coruña, Alicante y Málaga.
- **Ascensor** presenta la mayor beta secundaria en Ávila y Zaragoza.

Estos resultados muestran que la estructura de precios no es idéntica en todas las ciudades. En varios de los mercados analizados, la centralidad adquiere una importancia relativa destacada una vez controladas las demás variables incluidas en el modelo.

### 5.5 Multicolinealidad mediante VIF

El VIF evalúa cuánto aumenta la varianza estimada de un coeficiente debido a la relación lineal entre ese predictor y los demás predictores del modelo. Como guía práctica:

- valores próximos a 1: poca colinealidad;
- entre 5 y 10: colinealidad moderada que conviene vigilar;
- por encima de 10: posible problema serio.

In [ ]:
# =====================================================
# VIF DEL MODELO AMPLIADO
# =====================================================

vif_ampliado = (
    vif_todos[
        vif_todos["Modelo"].eq(
            "Ampliado"
        )
    ]
    .copy()
)

vif_max_ciudad = (
    vif_ampliado
    .groupby("Ciudad")["VIF"]
    .max()
    .sort_values(ascending=False)
    .rename("VIF máximo")
    .to_frame()
)

display(vif_max_ciudad.round(3))

display(
    vif_ampliado[
        vif_ampliado["VIF"] >= 5
    ]
    .sort_values(
        ["Ciudad", "VIF"],
        ascending=[True, False],
    )
    .round(3)
)

### Interpretación

No aparece ningún **VIF superior a 10**.

Los valores máximos se observan sobre todo en `size`:

- Madrid: aproximadamente **7,05**.
- Barcelona: aproximadamente **5,20**.
- Zaragoza: aproximadamente **5,19**.

La colinealidad observada es moderada y coherente con la relación natural entre superficie, habitaciones y baños. No se considera lo bastante alta como para retirar estas variables del modelo.

### 5.6 Significación estadística de las variables incorporadas

Se resume en cuántas ciudades resulta significativa cada variable adicional utilizando p-valores robustos HC3 y un nivel del 5 %.

In [ ]:
# =====================================================
# SIGNIFICANCIA DE VARIABLES ADICIONALES
# =====================================================

variables_adicionales = [
    "dist_center_km",
    "floor_num",
    "hasAirConditioning",
    "hasBoxRoom",
    "size_sq",
    "status_renew",
]

resumen_significancia = (
    coef_ampliado[
        coef_ampliado["Variable"]
        .isin(variables_adicionales)
    ]
    .groupby("Variable")
    .agg(
        Ciudades_estimadas=(
            "Ciudad", "nunique"
        ),
        Ciudades_significativa_5pct=(
            "Significativa_5pct_HC3",
            "sum",
        ),
    )
    .sort_values(
        "Ciudades_significativa_5pct",
        ascending=False,
    )
    .reset_index()
)

display(resumen_significancia)

### Interpretación

La variable adicional más consistente es **`dist_center_km`**, significativa en **10 de 10 ciudades**.

También destacan:

- `status_renew`: significativa en **9 de 10** ciudades.
- `hasAirConditioning`: **8 de 10**.
- `floor_num`: **7 de 10**.
- `size_sq`: **7 de 10**.
- `hasBoxRoom`: **4 de 10**.

Esto respalda la utilidad de incluir información de centralidad, estado y no linealidad de la superficie en la especificación ampliada.

### 5.7 Diagnóstico de residuos

Se revisan la prueba de Breusch-Pagan, que evalúa la presencia de heterocedasticidad, y la prueba de Jarque-Bera, que analiza posibles desviaciones de la normalidad residual.

Jarque-Bera puede rechazar la normalidad ante desviaciones pequeñas en muestras grandes, de modo que su resultado se interpreta con cautela y no se emplea por sí solo para valorar la adecuación del modelo. Los gráficos de residuos frente a valores ajustados se usan como complemento para detectar heterocedasticidad, curvatura o patrones anómalos, no como una prueba directa de normalidad.


In [ ]:
# =====================================================
# DIAGNÓSTICOS DEL MODELO AMPLIADO
# =====================================================

diagnosticos = (
    resultados_modelos[
        resultados_modelos["Modelo"]
        .eq("Ampliado")
    ]
    [
        [
            "Ciudad",
            "N",
            "R2_ajustado",
            "VIF_max",
            "Breusch_Pagan_p",
            "Jarque_Bera_p",
            "Variables_excluidas_varianza_cero",
        ]
    ]
    .sort_values("Ciudad")
)

display(diagnosticos.round(4))
# Figura 5-8: resumen visual de diagnósticos
tabla_fig_5_8 = diagnosticos.copy()
tabla_fig_5_8["R2_ajustado"] = tabla_fig_5_8["R2_ajustado"].map(
    lambda x: f"{x:.3f}"
)
tabla_fig_5_8["VIF_max"] = tabla_fig_5_8["VIF_max"].map(
    lambda x: f"{x:.3f}"
)
tabla_fig_5_8["Breusch_Pagan_p"] = tabla_fig_5_8["Breusch_Pagan_p"].map(
    lambda x: f"{x:.4f}"
)
tabla_fig_5_8["Jarque_Bera_p"] = tabla_fig_5_8["Jarque_Bera_p"].map(
    lambda x: f"{x:.4f}"
)
tabla_fig_5_8["Variables_excluidas_varianza_cero"] = (
    tabla_fig_5_8["Variables_excluidas_varianza_cero"]
    .replace("", "-")
)
tabla_fig_5_8 = tabla_fig_5_8.rename(
    columns={
        "R2_ajustado": "R² aj.",
        "VIF_max": "VIF máx.",
        "Breusch_Pagan_p": "BP p",
        "Jarque_Bera_p": "JB p",
        "Variables_excluidas_varianza_cero": "Excluidas (var. 0)",
    }
)

fig, ax = plt.subplots(figsize=(16, 6))
ax.axis("off")
tabla = ax.table(
    cellText=tabla_fig_5_8.values,
    colLabels=tabla_fig_5_8.columns,
    cellLoc="center",
    loc="center",
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(7.5)
tabla.scale(1, 1.35)
ax.set_title(
    "Resumen de los principales diagnósticos estadísticos",
    pad=16,
)
plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "figura_5_08_diagnosticos_modelos.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


### Interpretación

La prueba de Breusch-Pagan muestra **indicios de heterocedasticidad al nivel del 5 % en nueve de las diez ciudades**. Ávila es la única excepción, con un p-valor de 0,0891. La inferencia se presenta de forma sistemática mediante errores estándar robustos HC3.

La prueba de Jarque-Bera rechaza la hipótesis de normalidad residual al nivel del 5 % en nueve de las diez ciudades. Ávila vuelve a ser la única excepción, con un p-valor de 0,4789. Estos resultados se interpretan con cautela por el elevado tamaño muestral de varios mercados y se consideran un diagnóstico complementario, sin utilizar el contraste de normalidad como criterio aislado para descartar los modelos. Los gráficos de residuos frente a valores ajustados se analizan por separado para detectar patrones de varianza, curvatura o grupos de observaciones anómalas.


## 5.8 Validación fuera de muestra mediante train/test

Los apartados anteriores tienen una finalidad principalmente **explicativa e inferencial**: los modelos OLS se estiman con la muestra analítica completa de cada ciudad para comparar coeficientes beta estandarizados, significación robusta HC3, R² ajustado, AIC/BIC y diagnósticos.

Como comprobación adicional de la **capacidad de generalización**, se incorpora una validación fuera de muestra. Para cada ciudad se realiza una partición reproducible del **80 % para entrenamiento y 20 % para prueba**, utilizando `random_state=42`.

Para evitar *data leakage*, las transformaciones que dependen de parámetros de los datos se estiman exclusivamente con el conjunto de entrenamiento y después se aplican al conjunto de prueba. Esto afecta a la imputación por mediana, al centrado utilizado para `size_sq` y a la estandarización Z-score. Las reglas deterministas —por ejemplo, la codificación de `status_renew`, las variables binarias sin imputación estadística y la distancia al punto central de referencia— se mantienen iguales en ambos conjuntos.

La evaluación se realiza sobre observaciones que no intervienen en el ajuste. Se presentan tres métricas sobre `log_price` en su escala original logarítmica: **R² de test**, **RMSE** y **MAE**. Un R² de test mayor indica una mayor proporción de variabilidad explicada fuera de muestra; valores menores de RMSE y MAE indican menor error de predicción.


In [ ]:
# =====================================================
# FUNCIONES PARA VALIDACIÓN TRAIN/TEST SIN DATA LEAKAGE
# =====================================================

RANDOM_STATE = 42
TEST_SIZE = 0.20


def preparar_variables_validacion(train, test):
    """Prepara train y test usando parámetros aprendidos solo en train."""
    train = train.copy()
    test = test.copy()

    # Variable objetivo
    train["log_price"] = np.log(train["price"])
    test["log_price"] = np.log(test["price"])

    # Binarias con imputación por mediana: la mediana se aprende solo en train.
    mapeo_binarias_validacion = {
        "hasLift": "hasLift",
        "exterior": "exterior",
        "features/hasTerrace": "hasTerrace",
        "features/hasSwimmingPool": "hasSwimmingPool",
        "features/hasGarden": "hasGarden",
        "features/hasAirConditioning": "hasAirConditioning",
        "features/hasBoxRoom": "hasBoxRoom",
    }

    for origen, destino in mapeo_binarias_validacion.items():
        train_raw = convertir_binaria(train[origen])
        test_raw = convertir_binaria(test[origen])

        train[f"{destino}_missing"] = train_raw.isna().astype(int)
        test[f"{destino}_missing"] = test_raw.isna().astype(int)

        mediana_train = train_raw.median()
        if pd.isna(mediana_train):
            mediana_train = 0.0

        train[destino] = train_raw.fillna(mediana_train).astype(float)
        test[destino] = test_raw.fillna(mediana_train).astype(float)

    # Estas dos variables siguen la regla determinista del modelo original.
    train["hasParking"] = (
        convertir_binaria(train["parkingSpace/hasParkingSpace"])
        .fillna(0)
        .astype(float)
    )
    test["hasParking"] = (
        convertir_binaria(test["parkingSpace/hasParkingSpace"])
        .fillna(0)
        .astype(float)
    )

    train["newDevelopment"] = (
        convertir_binaria(train["newDevelopment"])
        .fillna(0)
        .astype(float)
    )
    test["newDevelopment"] = (
        convertir_binaria(test["newDevelopment"])
        .fillna(0)
        .astype(float)
    )

    # Estado de conservación.
    for datos in (train, test):
        status_norm = (
            datos["status"]
            .astype("string")
            .str.lower()
            .str.strip()
        )
        datos["status_renew"] = (
            status_norm.eq("renew").fillna(False).astype(int)
        )
        datos["status_missing"] = datos["status"].isna().astype(int)

    # Planta: la mediana utilizada para imputar se aprende solo en train.
    train["floor_num_raw"] = train["floor"].apply(convertir_planta)
    test["floor_num_raw"] = test["floor"].apply(convertir_planta)

    train["floor_missing"] = train["floor_num_raw"].isna().astype(int)
    test["floor_missing"] = test["floor_num_raw"].isna().astype(int)

    mediana_floor_train = train["floor_num_raw"].median()
    if pd.isna(mediana_floor_train):
        mediana_floor_train = 0.0

    train["floor_num"] = train["floor_num_raw"].fillna(mediana_floor_train)
    test["floor_num"] = test["floor_num_raw"].fillna(mediana_floor_train)

    # Distancia al centro: transformación determinista, sin parámetros aprendidos.
    ciudad = train["ciudad"].iloc[0]
    lat0, lon0 = CENTROS_CIUDAD[ciudad]
    train["dist_center_km"] = haversine_km(
        train["latitude"], train["longitude"], lat0, lon0
    )
    test["dist_center_km"] = haversine_km(
        test["latitude"], test["longitude"], lat0, lon0
    )

    # Término cuadrático: el centro se aprende exclusivamente en train.
    media_size_train = train["size"].mean()
    train["size_sq"] = (train["size"] - media_size_train) ** 2
    test["size_sq"] = (test["size"] - media_size_train) ** 2

    return train, test


def ajustar_validacion_train_test(df_ciudad, ciudad, variables, nombre_modelo):
    """Ajusta OLS en train y evalúa R², RMSE y MAE en test."""
    train, test = train_test_split(
        df_ciudad,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    train, test = preparar_variables_validacion(train, test)

    variables_presentes = [v for v in variables if v in train.columns]

    # Las variables sin variación se identifican únicamente en train.
    variables_constantes = [
        v for v in variables_presentes
        if train[v].nunique(dropna=True) <= 1
    ]
    variables_utilizadas = [
        v for v in variables_presentes
        if v not in variables_constantes
    ]

    columnas_uso = variables_utilizadas + ["log_price"]
    train[columnas_uso] = train[columnas_uso].replace([np.inf, -np.inf], np.nan)
    test[columnas_uso] = test[columnas_uso].replace([np.inf, -np.inf], np.nan)
    train = train.dropna(subset=columnas_uso).copy()
    test = test.dropna(subset=columnas_uso).copy()

    X_train = train[variables_utilizadas].astype(float)
    X_test = test[variables_utilizadas].astype(float)
    y_train = train["log_price"].astype(float)
    y_test = test["log_price"].astype(float)

    # Parámetros de estandarización aprendidos exclusivamente en train.
    media_X = X_train.mean()
    std_X = X_train.std(ddof=0)
    if (std_X == 0).any():
        raise ValueError(f"{ciudad} - {nombre_modelo}: variable sin variación en train.")

    media_y = y_train.mean()
    std_y = y_train.std(ddof=0)
    if std_y == 0:
        raise ValueError(f"{ciudad}: log_price no presenta variación en train.")

    X_train_z = (X_train - media_X) / std_X
    X_test_z = (X_test - media_X) / std_X
    y_train_z = (y_train - media_y) / std_y

    X_train_ols = sm.add_constant(X_train_z, has_constant="add")
    X_test_ols = sm.add_constant(X_test_z, has_constant="add")
    X_test_ols = X_test_ols.reindex(columns=X_train_ols.columns, fill_value=0)

    modelo = sm.OLS(y_train_z, X_train_ols).fit()
    pred_test_z = modelo.predict(X_test_ols)

    # Se devuelve la predicción a la escala logarítmica original.
    pred_test_log = pred_test_z * std_y + media_y

    return {
        "Ciudad": ciudad,
        "Modelo": nombre_modelo,
        "N_train": len(train),
        "N_test": len(test),
        "N_variables": len(variables_utilizadas),
        "R2_train": modelo.rsquared,
        "R2_test": r2_score(y_test, pred_test_log),
        "RMSE_test_log": np.sqrt(mean_squared_error(y_test, pred_test_log)),
        "MAE_test_log": mean_absolute_error(y_test, pred_test_log),
        "Variables_excluidas_varianza_cero": ", ".join(variables_constantes),
    }


### 5.8.1 Ajuste y evaluación por ciudad

Se aplica la misma partición 80/20 a los modelos base y ampliado de cada ciudad. Al utilizar el mismo `random_state`, ambos modelos se evalúan sobre las mismas observaciones de prueba dentro de cada mercado, lo que permite una comparación directa de sus métricas fuera de muestra.


In [ ]:
# =====================================================
# VALIDACIÓN TRAIN/TEST DE LOS MODELOS LINEALES
# =====================================================

resultados_validacion = []

for ciudad, df_ciudad in df_modelo.groupby("ciudad", sort=True):
    especificaciones = {
        "Base": VARIABLES_BASE,
        "Ampliado": VARIABLES_AMPLIADAS,
    }

    for nombre_modelo, variables in especificaciones.items():
        resultado = ajustar_validacion_train_test(
            df_ciudad=df_ciudad,
            ciudad=ciudad,
            variables=variables,
            nombre_modelo=nombre_modelo,
        )
        resultados_validacion.append(resultado)

resultados_validacion = pd.DataFrame(resultados_validacion)

display(
    resultados_validacion[
        [
            "Ciudad", "Modelo", "N_train", "N_test",
            "R2_train", "R2_test", "RMSE_test_log", "MAE_test_log",
        ]
    ].round(4)
)


### Interpretación

La separación entre entrenamiento y prueba permite comprobar si el comportamiento de los modelos se mantiene con observaciones que no han participado en el ajuste. En general, los valores de R² obtenidos en test son próximos a los de entrenamiento, especialmente en las ciudades con mayor número de observaciones.

En el modelo ampliado, Madrid obtiene un R² de 0,8329 en entrenamiento y 0,8291 en test; Barcelona pasa de 0,8115 a 0,8225 y Zaragoza de 0,7853 a 0,8159. También se observa un comportamiento estable en Valencia y Las Palmas. En Sevilla y Málaga el R² de test es algo inferior al de entrenamiento, aunque se mantiene en niveles próximos.

Ávila presenta la diferencia más acusada: el modelo ampliado pasa de un R² de 0,6751 en entrenamiento a 0,3722 en test. Este resultado debe interpretarse con prudencia, ya que la partición deja únicamente 158 observaciones para entrenamiento y 40 para prueba. En conjunto, la validación no muestra una pérdida generalizada de rendimiento fuera de muestra, aunque confirma que los resultados de Ávila son menos estables debido al reducido tamaño de su muestra.


### 5.8.2 Comparación fuera de muestra entre modelo base y ampliado

Para facilitar la comparación se reorganizan las métricas de test por ciudad. La diferencia de R² se calcula como `R² test ampliado - R² test base`; por tanto, un valor positivo indica que la especificación ampliada explica una mayor proporción de variabilidad en observaciones no utilizadas durante el entrenamiento.


In [ ]:
# =====================================================
# COMPARACIÓN BASE VS. AMPLIADO EN TEST
# =====================================================

comparacion_test = resultados_validacion.pivot(
    index="Ciudad",
    columns="Modelo",
    values=["R2_test", "RMSE_test_log", "MAE_test_log"],
)

comparacion_test.columns = [
    f"{metrica}_{modelo}"
    for metrica, modelo in comparacion_test.columns
]

comparacion_test = comparacion_test.reset_index()
comparacion_test["Mejora_R2_test"] = (
    comparacion_test["R2_test_Ampliado"]
    - comparacion_test["R2_test_Base"]
)
comparacion_test["Reduccion_RMSE_test_log"] = (
    comparacion_test["RMSE_test_log_Base"]
    - comparacion_test["RMSE_test_log_Ampliado"]
)
comparacion_test["Reduccion_MAE_test_log"] = (
    comparacion_test["MAE_test_log_Base"]
    - comparacion_test["MAE_test_log_Ampliado"]
)

comparacion_test = comparacion_test.sort_values(
    "Mejora_R2_test",
    ascending=False,
)

display(comparacion_test.round(4))


### Interpretación

El modelo ampliado obtiene un R² de test superior al modelo base en las diez ciudades. Las diferencias más claras aparecen en Sevilla (+0,1723) y Las Palmas (+0,1661), seguidas de Madrid (+0,0820), Barcelona (+0,0819), Valencia (+0,0641) y Zaragoza (+0,0520). En Málaga, Alicante y A Coruña la mejora es más moderada.

Además del aumento del R², el RMSE y el MAE disminuyen en prácticamente todos los mercados. La excepción es Ávila: el R² apenas cambia, de 0,3719 a 0,3722, y el MAE aumenta ligeramente de 0,1783 a 0,1820. Por tanto, en esta ciudad no puede afirmarse que la especificación ampliada aporte una mejora predictiva clara.

En el resto de ciudades, los resultados fuera de muestra siguen el mismo patrón observado anteriormente con el R² ajustado. Esto aporta una comprobación adicional de que las variables incorporadas al modelo ampliado mejoran el ajuste sin que esa mejora quede limitada únicamente a los datos utilizados para estimar la regresión.


### 5.8.3 Visualización del R² fuera de muestra

Se representa el R² de test de ambos modelos para cada ciudad. Esta figura complementa la comparación del R² ajustado de la muestra completa y permite distinguir visualmente entre **calidad de ajuste** y **rendimiento fuera de muestra**.


In [ ]:
# =====================================================
# R² DE TEST POR CIUDAD
# =====================================================

plot_test = resultados_validacion.pivot(
    index="Ciudad",
    columns="Modelo",
    values="R2_test",
).sort_index()

ax = plot_test.plot(
    kind="bar",
    figsize=(12, 6),
)
ax.set_title("R² fuera de muestra por ciudad")
ax.set_xlabel("Ciudad")
ax.set_ylabel("R² de test")
ax.legend(title="Modelo")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Interpretación

La representación gráfica confirma que el modelo ampliado obtiene un R² de test superior al modelo base en todas las ciudades. Las diferencias son especialmente visibles en Sevilla y Las Palmas, mientras que en Ávila ambos modelos presentan prácticamente el mismo resultado. El gráfico facilita así la comparación de la capacidad de generalización de las dos especificaciones y complementa las métricas mostradas en la tabla anterior.


## 5.9 Modelo alternativo: Random Forest

Como complemento a la regresión lineal múltiple se incorpora un **Random Forest Regressor**, un algoritmo de aprendizaje automático basado en un conjunto de árboles de decisión. Su finalidad en este trabajo no es sustituir al modelo OLS, que continúa siendo la herramienta principal para interpretar las asociaciones entre las características de la vivienda y el precio de oferta, sino disponer de una **referencia predictiva no lineal**.

Random Forest puede capturar relaciones no lineales e interacciones entre variables sin necesidad de especificarlas explícitamente. Para que la comparación sea homogénea, se utiliza la misma división **80 % entrenamiento / 20 % prueba**, el mismo `random_state=42` y las mismas variables del modelo ampliado. Toda transformación que requiere estimar un parámetro se obtiene exclusivamente a partir del conjunto de entrenamiento, evitando así *data leakage*.

El modelo se estima con 500 árboles (`n_estimators=500`) y una semilla fija para garantizar la reproducibilidad. No se realiza una optimización de hiperparámetros sobre el conjunto de test, ya que este se reserva exclusivamente para la evaluación final. El rendimiento se mide mediante **R², RMSE y MAE sobre `log_price`**, las mismas métricas utilizadas en la validación fuera de muestra del modelo lineal.


In [ ]:
# =====================================================
# RANDOM FOREST: AJUSTE Y EVALUACIÓN POR CIUDAD
# =====================================================

from sklearn.ensemble import RandomForestRegressor


def ajustar_random_forest(df_ciudad, ciudad, variables):
    """Ajusta Random Forest en train y evalúa R², RMSE y MAE en test."""
    train, test = train_test_split(
        df_ciudad,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    # Se reutiliza la preparación definida en 5.8.
    # Los parámetros necesarios se aprenden únicamente con train.
    train, test = preparar_variables_validacion(train, test)

    variables_presentes = [v for v in variables if v in train.columns]
    variables_constantes = [
        v for v in variables_presentes
        if train[v].nunique(dropna=True) <= 1
    ]
    variables_utilizadas = [
        v for v in variables_presentes
        if v not in variables_constantes
    ]

    columnas_uso = variables_utilizadas + ["log_price"]
    train[columnas_uso] = train[columnas_uso].replace([np.inf, -np.inf], np.nan)
    test[columnas_uso] = test[columnas_uso].replace([np.inf, -np.inf], np.nan)
    train = train.dropna(subset=columnas_uso).copy()
    test = test.dropna(subset=columnas_uso).copy()

    X_train = train[variables_utilizadas].astype(float)
    X_test = test[variables_utilizadas].astype(float)
    y_train = train["log_price"].astype(float)
    y_test = test["log_price"].astype(float)

    modelo_rf = RandomForestRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    modelo_rf.fit(X_train, y_train)

    pred_train = modelo_rf.predict(X_train)
    pred_test = modelo_rf.predict(X_test)

    resultado = {
        "Ciudad": ciudad,
        "Modelo": "Random Forest",
        "N_train": len(train),
        "N_test": len(test),
        "N_variables": len(variables_utilizadas),
        "R2_train": r2_score(y_train, pred_train),
        "R2_test": r2_score(y_test, pred_test),
        "RMSE_test_log": np.sqrt(mean_squared_error(y_test, pred_test)),
        "MAE_test_log": mean_absolute_error(y_test, pred_test),
        "Variables_excluidas_varianza_cero": ", ".join(variables_constantes),
    }

    importancias = pd.DataFrame({
        "Ciudad": ciudad,
        "Variable": variables_utilizadas,
        "Importancia_RF": modelo_rf.feature_importances_,
    }).sort_values("Importancia_RF", ascending=False)

    return resultado, importancias


resultados_rf = []
importancias_rf = []

for ciudad, df_ciudad in df_modelo.groupby("ciudad", sort=True):
    resultado_rf, importancia_rf = ajustar_random_forest(
        df_ciudad=df_ciudad,
        ciudad=ciudad,
        variables=VARIABLES_AMPLIADAS,
    )
    resultados_rf.append(resultado_rf)
    importancias_rf.append(importancia_rf)

resultados_rf = pd.DataFrame(resultados_rf)
importancias_rf = pd.concat(importancias_rf, ignore_index=True)

display(
    resultados_rf[
        [
            "Ciudad", "Modelo", "N_train", "N_test",
            "R2_train", "R2_test", "RMSE_test_log", "MAE_test_log",
        ]
    ].round(4)
)


### Interpretación

Random Forest obtiene valores de R² de test comprendidos entre 0,5282 en Ávila y 0,8761 en Madrid. También alcanza valores elevados en Barcelona (0,8640), Zaragoza (0,8483), Alicante (0,8275) y Sevilla (0,8184).

Los R² de entrenamiento son claramente superiores y se sitúan entre 0,9461 y 0,9841. Esta diferencia muestra que el modelo se ajusta con mucha intensidad a los datos de entrenamiento, algo esperable en un método flexible basado en árboles. Por este motivo, la valoración del modelo se realiza a partir de las métricas obtenidas en test y no de su ajuste sobre la muestra de entrenamiento.

A pesar de esa diferencia, los resultados de test mantienen un nivel elevado en la mayoría de las ciudades. Ávila vuelve a presentar el resultado más bajo y debe analizarse con cautela por el reducido número de observaciones disponibles.


### 5.9.1 Comparación OLS ampliado vs. Random Forest

La comparación se realiza exclusivamente sobre el conjunto de prueba. Ambos modelos utilizan la misma partición de observaciones y las mismas variables explicativas, de modo que las diferencias en las métricas reflejan el comportamiento de cada enfoque sobre datos no utilizados durante el ajuste.

Una diferencia positiva en `Diferencia_R2_RF_menos_OLS` indica un R² de test superior para Random Forest. En RMSE y MAE se calcula la reducción del error respecto al OLS, por lo que los valores positivos indican menor error para Random Forest.


In [ ]:
# =====================================================
# COMPARACIÓN OLS AMPLIADO VS. RANDOM FOREST EN TEST
# =====================================================

ols_ampliado_test = (
    resultados_validacion
    .loc[
        resultados_validacion["Modelo"].eq("Ampliado"),
        ["Ciudad", "R2_test", "RMSE_test_log", "MAE_test_log"],
    ]
    .rename(columns={
        "R2_test": "R2_test_OLS",
        "RMSE_test_log": "RMSE_test_log_OLS",
        "MAE_test_log": "MAE_test_log_OLS",
    })
)

rf_test = (
    resultados_rf[
        ["Ciudad", "R2_test", "RMSE_test_log", "MAE_test_log"]
    ]
    .rename(columns={
        "R2_test": "R2_test_RF",
        "RMSE_test_log": "RMSE_test_log_RF",
        "MAE_test_log": "MAE_test_log_RF",
    })
)

comparacion_ols_rf = ols_ampliado_test.merge(
    rf_test,
    on="Ciudad",
    how="inner",
)

comparacion_ols_rf["Diferencia_R2_RF_menos_OLS"] = (
    comparacion_ols_rf["R2_test_RF"]
    - comparacion_ols_rf["R2_test_OLS"]
)
comparacion_ols_rf["Reduccion_RMSE_RF"] = (
    comparacion_ols_rf["RMSE_test_log_OLS"]
    - comparacion_ols_rf["RMSE_test_log_RF"]
)
comparacion_ols_rf["Reduccion_MAE_RF"] = (
    comparacion_ols_rf["MAE_test_log_OLS"]
    - comparacion_ols_rf["MAE_test_log_RF"]
)

comparacion_ols_rf = comparacion_ols_rf.sort_values(
    "Diferencia_R2_RF_menos_OLS",
    ascending=False,
)

display(comparacion_ols_rf.round(4))


### Interpretación

Random Forest mejora las tres métricas de evaluación frente al OLS ampliado en las diez ciudades: aumenta el R² de test y reduce tanto el RMSE como el MAE. La diferencia de R² es especialmente elevada en Alicante (+0,1950), Ávila (+0,1560), Málaga (+0,1366), Sevilla (+0,1327), A Coruña (+0,1063) y Las Palmas (+0,1006). En Valencia la mejora es de 0,0595, mientras que en Madrid (+0,0469), Barcelona (+0,0415) y Zaragoza (+0,0324) es menor porque el modelo lineal ampliado ya presenta un R² de test elevado.

El mismo patrón aparece en los errores de predicción. Por ejemplo, en Alicante el RMSE pasa de 0,3407 con OLS a 0,2334 con Random Forest y el MAE de 0,2640 a 0,1617. En Sevilla el RMSE disminuye de 0,3641 a 0,2768 y el MAE de 0,2627 a 0,1949.

Estos resultados indican que una especificación más flexible recoge relaciones no lineales e interacciones que no quedan completamente representadas por la regresión lineal. Sin embargo, el objetivo principal del trabajo sigue siendo analizar e interpretar las asociaciones entre las características de las viviendas y el precio. Por ello, OLS se mantiene como modelo principal y Random Forest se utiliza como contraste predictivo complementario.


### 5.9.2 Visualización comparativa del R² de test

La siguiente figura representa el R² obtenido fuera de muestra por el modelo OLS ampliado y Random Forest en cada ciudad, facilitando la comparación visual de ambos enfoques.


In [ ]:
# =====================================================
# R² DE TEST: OLS AMPLIADO VS. RANDOM FOREST
# =====================================================

plot_ols_rf = comparacion_ols_rf.set_index("Ciudad")[
    ["R2_test_OLS", "R2_test_RF"]
].sort_index()

ax = plot_ols_rf.plot(
    kind="bar",
    figsize=(12, 6),
)
ax.set_title("R² fuera de muestra: OLS ampliado vs. Random Forest")
ax.set_xlabel("Ciudad")
ax.set_ylabel("R² de test")
ax.legend(["OLS ampliado", "Random Forest"], title="Modelo")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
ax.figure.savefig(
    FIGURES_DIR / "figura_5_11_r2_test_ols_random_forest.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


### 5.9.3 Importancia de variables en Random Forest

Como información complementaria se muestran las cinco variables con mayor importancia interna del Random Forest en cada ciudad. Esta medida refleja cuánto contribuye cada variable a las reducciones de impureza acumuladas por los árboles y **no debe interpretarse como un coeficiente, un efecto causal ni una magnitud directamente comparable con las betas estandarizadas del OLS**. Su función es únicamente descriptiva dentro del modelo predictivo alternativo.


In [ ]:
# =====================================================
# CINCO VARIABLES CON MAYOR IMPORTANCIA POR CIUDAD
# =====================================================

top5_importancias_rf = (
    importancias_rf
    .sort_values(["Ciudad", "Importancia_RF"], ascending=[True, False])
    .groupby("Ciudad", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

display(top5_importancias_rf.round(4))


### Interpretación

Las importancias de Random Forest muestran algunos patrones que coinciden con los obtenidos mediante la regresión, aunque ambas medidas tienen un significado diferente. La superficie (`size`) ocupa la primera posición en A Coruña, Ávila, Barcelona, Madrid, Sevilla y Valencia. En Alicante, Málaga y Zaragoza la variable con mayor importancia es el número de baños, mientras que en Las Palmas destaca la distancia al punto central de referencia.

La distancia al centro aparece entre las variables más relevantes en las diez ciudades y ocupa la primera posición en Las Palmas. También se observa una presencia frecuente de `size_sq`, lo que es coherente con la posibilidad de que la relación entre superficie y precio no sea estrictamente lineal.

Estos resultados sirven como apoyo al análisis principal, pero no sustituyen la interpretación de las betas estandarizadas. La importancia de una variable en Random Forest no informa sobre el signo de la asociación ni puede interpretarse como un efecto sobre el precio. Por este motivo, las conclusiones sobre dirección, magnitud relativa e inferencia estadística continúan basándose en los modelos OLS.


## 6. Visualización comparativa de resultados

### 6.1 R² ajustado por ciudad

In [ ]:
# =====================================================
# COMPARACIÓN GRÁFICA DEL R² AJUSTADO
# =====================================================

plot_r2 = (
    comparacion[
        [
            "Ciudad",
            "R2_ajustado_Base",
            "R2_ajustado_Ampliado",
        ]
    ]
    .set_index("Ciudad")
)

ax = plot_r2.plot(
    kind="bar",
    figsize=(12, 6),
)

ax.set_title(
    "R² ajustado: modelo base y modelo ampliado"
)
ax.set_xlabel("Ciudad")
ax.set_ylabel("R² ajustado")
ax.set_ylim(0, 0.9)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
ax.figure.savefig(
    FIGURES_DIR / "figura_5_02_r2_ajustado_modelos.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

### Interpretación

El gráfico confirma visualmente que el modelo ampliado presenta un mayor R² ajustado en las diez ciudades. La mejora resulta especialmente visible en Sevilla y Las Palmas.

### 6.2 Mapa de calor de betas estandarizadas

El mapa de calor conserva el signo de los coeficientes, de modo que los valores positivos y negativos pueden compararse visualmente entre ciudades.

In [ ]:
# =====================================================
# MATRIZ Y HEATMAP DE BETAS
# =====================================================

matriz_betas = (
    coef_sustantivos
    .pivot(
        index="Ciudad",
        columns="Variable",
        values="Beta_estandarizada",
    )
)

display(
    matriz_betas.round(3)
)

plt.figure(figsize=(16, 7))

sns.heatmap(
    matriz_betas,
    annot=True,
    fmt=".2f",
    center=0,
    cmap="coolwarm",
    linewidths=0.4,
)

plt.title(
    "Coeficientes beta estandarizados por ciudad"
)
plt.xlabel("Variable")
plt.ylabel("Ciudad")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figura_5_04_betas_estandarizadas.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

### Interpretación

El mapa muestra con claridad el predominio de `size` y permite observar el signo negativo de `dist_center_km` en las diez ciudades.

También permite identificar que la magnitud de baños, ascensor u otros equipamientos cambia entre mercados, lo que refuerza la utilidad de estimar modelos separados.

### 6.3 Diagramas radar

Los radares utilizan `|beta|` porque la distancia radial no puede ser negativa. Por ello deben leerse junto al mapa de calor, que conserva el signo.

Se genera una versión sin `size` y `size_sq` para que la superficie no oculte las diferencias entre el resto de variables.

In [ ]:
# =====================================================
# RADARES SIN SUPERFICIE
# =====================================================

matriz_radar = (
    matriz_betas
    .drop(
        columns=["size", "size_sq"],
        errors="ignore",
    )
    .abs()
)

variables_radar = matriz_radar.columns.tolist()
n_variables = len(variables_radar)

angulos = np.linspace(
    0,
    2 * np.pi,
    n_variables,
    endpoint=False,
).tolist()
angulos += angulos[:1]

ciudades = matriz_radar.index.tolist()

n_cols = 2
n_rows = int(
    np.ceil(len(ciudades) / n_cols)
)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 4.8 * n_rows),
    subplot_kw={"polar": True},
)

axes = np.array(axes).ravel()

for ax, ciudad in zip(axes, ciudades):
    valores = (
        matriz_radar
        .loc[ciudad]
        .fillna(0)
        .tolist()
    )
    valores += valores[:1]

    ax.plot(
        angulos,
        valores,
        linewidth=2,
    )
    ax.fill(
        angulos,
        valores,
        alpha=0.15,
    )

    ax.set_xticks(
        angulos[:-1]
    )
    ax.set_xticklabels(
        variables_radar,
        fontsize=8,
    )
    ax.set_title(
        ciudad,
        pad=18,
    )

for ax in axes[len(ciudades):]:
    ax.remove()

fig.suptitle(
    "Importancia relativa por ciudad "
    "(|beta|, excluyendo superficie)",
    y=1.01,
    fontsize=15,
)

plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "figura_5_05_radar_sin_superficie.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

### Interpretación
Los radares permiten comparar el perfil de cada ciudad excluyendo las variables relacionadas con la superficie, cuya elevada magnitud podría dificultar la visualización de las diferencias entre los restantes factores. Se observan perfiles diferenciados: la distancia al punto central de referencia presenta la mayor importancia relativa entre los factores secundarios en Barcelona, Las Palmas, Madrid, Sevilla y Valencia; el número de baños destaca en A Coruña, Alicante y Málaga; y el ascensor en Ávila y Zaragoza.

El radar funciona como apoyo visual; las conclusiones deben basarse en las tablas de betas, sus signos y sus p-valores.

### 6.4 Residuos frente a valores ajustados

Se representan los residuos del modelo ampliado para detectar patrones de varianza, curvatura o grupos de observaciones anómalas.

In [ ]:
# =====================================================
# RESIDUOS FRENTE A PREDICCIONES
# =====================================================

ciudades = sorted(modelos.keys())

fig, axes = plt.subplots(
    5,
    2,
    figsize=(14, 20),
)

axes = axes.ravel()

for ax, ciudad in zip(axes, ciudades):
    modelo = modelos[ciudad]["Ampliado"]

    ax.scatter(
        modelo.fittedvalues,
        modelo.resid,
        alpha=0.25,
        s=12,
    )

    ax.axhline(
        0,
        linewidth=1,
    )

    ax.set_title(ciudad)
    ax.set_xlabel("Valor ajustado")
    ax.set_ylabel("Residuo")

plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "figura_5_07_residuos_ajustados.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

### Interpretación

Los gráficos muestran patrones compatibles con una dispersión residual no completamente constante en varios mercados, en línea con la prueba de Breusch-Pagan. Este resultado es coherente con el uso sistemático de errores estándar robustos HC3 para la inferencia.

No se aprecia un único patrón común a todas las ciudades, lo que vuelve a reflejar diferencias entre mercados inmobiliarios.


## 7. Exportación de resultados

Se guardan las tablas principales para facilitar la redacción de la memoria y evitar tener que reajustar los modelos cada vez que se necesite una cifra.

In [ ]:
# =====================================================
# EXPORTACIÓN DE RESULTADOS
# =====================================================

resumen_muestra.to_csv(
    TABLES_DIR
    / "muestra_por_ciudad.csv",
    index=False,
)

resultados_modelos.to_csv(
    TABLES_DIR
    / "resumen_modelos_por_ciudad.csv",
    index=False,
)

comparacion.to_csv(
    TABLES_DIR
    / "comparacion_modelos.csv",
    index=False,
)

coeficientes.to_csv(
    TABLES_DIR
    / "coeficientes_estandarizados_HC3.csv",
    index=False,
)

vif_todos.to_csv(
    TABLES_DIR
    / "vif_por_ciudad.csv",
    index=False,
)

mas_influyente_sin_size.to_csv(
    TABLES_DIR
    / "variable_mas_influyente_sin_superficie.csv",
    index=False,
)

resumen_significancia.to_csv(
    TABLES_DIR
    / "significancia_variables_adicionales.csv",
    index=False,
)

resultados_validacion.to_csv(
    TABLES_DIR / "validacion_train_test_modelos.csv",
    index=False,
)

comparacion_test.to_csv(
    TABLES_DIR / "comparacion_train_test_modelos.csv",
    index=False,
)

print(
    f"Tablas guardadas en: "
    f"{TABLES_DIR}"
)

### Interpretación

Las tablas exportadas contienen las cifras principales que deberán trasladarse a los apartados de resultados y discusión del TFM. Conviene utilizar estos archivos como fuente de las cifras de la memoria para evitar errores de transcripción.

## 8. Resultados principales

- Se modelizan **42.441 pisos** distribuidos en diez ciudades.
- La superficie es el predictor con mayor beta absoluta en todos los mercados.
- El modelo ampliado mejora el R² ajustado en **10 de 10 ciudades**.
- Los R² ajustados del modelo ampliado oscilan aproximadamente entre **0,61 y 0,83**.
- Madrid (**0,832**), Barcelona (**0,813**) y Zaragoza (**0,790**) presentan los mayores R² ajustados.
- La mejora más elevada se observa en Sevilla (**+0,170**) y Las Palmas (**+0,167**).
- Excluyendo superficie, `dist_center_km` domina en Barcelona, Las Palmas, Madrid, Sevilla y Valencia.
- Baños domina en A Coruña, Alicante y Málaga; ascensor en Ávila y Zaragoza.
- `dist_center_km` es estadísticamente significativa en **10/10 ciudades** y presenta beta negativa en todas.
- No se detectan VIF superiores a 10; la multicolinealidad más elevada es moderada y se concentra principalmente en `size`.
- La prueba de Breusch-Pagan muestra indicios de heterocedasticidad en 9 de las 10 ciudades; Ávila es la única excepción al nivel de significación del 5 %. Por este motivo, la inferencia se presenta mediante errores estándar robustos HC3 en todos los modelos.

> **Validación añadida:** los resultados train/test deben incorporarse a este resumen una vez ejecutado el notebook y comprobadas las métricas fuera de muestra.


## 9. Conclusiones de la fase

La modelización muestra que la estructura de asociaciones con el precio de oferta presenta diferencias entre ciudades y que no puede resumirse mediante una única característica.

La **superficie** es el predictor con mayor beta absoluta en las diez ciudades. Sin embargo, cuando se excluye la superficie para observar mejor los factores secundarios, la importancia relativa cambia entre mercados. La **distancia al punto central de referencia** destaca en Barcelona, Las Palmas, Madrid, Sevilla y Valencia; el **número de baños** en A Coruña, Alicante y Málaga; y la disponibilidad de **ascensor** en Ávila y Zaragoza.

La especificación ampliada presenta un R² ajustado superior en las diez ciudades y mantiene niveles de multicolinealidad que no alcanzan el umbral de VIF 10. Los indicios de heterocedasticidad observados en nueve mercados respaldan el uso sistemático de errores estándar robustos HC3 para la inferencia.

La validación train/test muestra además que el modelo ampliado obtiene un R² de test superior al modelo base en las diez ciudades. Las mejoras son especialmente claras en Sevilla y Las Palmas. Ávila constituye la principal excepción en términos prácticos, ya que la diferencia entre ambas especificaciones es mínima y la muestra de test contiene únicamente 40 observaciones.

Como comprobación adicional se ha estimado un Random Forest con las mismas variables y la misma partición de entrenamiento y prueba. El modelo mejora R², RMSE y MAE frente al OLS ampliado en las diez ciudades, lo que sugiere que existen relaciones no lineales e interacciones que la especificación lineal no recoge por completo. Aun así, Random Forest se mantiene como referencia predictiva complementaria. El OLS continúa siendo el modelo principal del trabajo porque permite comparar el signo y la magnitud de las asociaciones mediante betas estandarizadas y realizar inferencia con errores robustos HC3.

### Limitaciones que deben recogerse en la memoria

1. Los datos corresponden a **precios de oferta de anuncios**, no a precios finales de compraventa.
2. El análisis es **observacional**; los coeficientes representan asociaciones y no efectos causales.
3. La distancia al punto central de referencia es una aproximación operativa a la centralidad y no recoge barrio, costa, accesibilidad, servicios o calidad urbana.
4. Algunas características contienen valores no informados y requieren un tratamiento específico para conservar la muestra.
5. Ávila cuenta con una muestra considerablemente menor que el resto de ciudades, circunstancia que también se refleja en la validación fuera de muestra.
6. Pueden existir factores relevantes no incluidos en el modelo y dependencia espacial entre viviendas próximas.
7. Random Forest se ha utilizado como contraste predictivo con una configuración reproducible, sin realizar una búsqueda exhaustiva de hiperparámetros. Su finalidad es comprobar el comportamiento de un modelo alternativo y no optimizar una solución predictiva final.


## 10. Conclusiones finales del análisis

En conjunto, los resultados permiten dar respuesta al objetivo propuesto: **las características asociadas al precio no presentan la misma importancia relativa en todas las ciudades**.

La superficie es el factor más estable y dominante, pero la localización y ciertas características internas adquieren una importancia relativa distinta según el mercado. La distancia al punto central de referencia resulta especialmente importante en varios de los mercados analizados, mientras que en otros destacan variables como los baños o el ascensor.

Los modelos ampliados alcanzan valores de R² ajustado situados aproximadamente entre **0,61 y 0,83**, lo que indica que el conjunto de variables empleado explica una parte importante de la variabilidad observada en `log_price`, aunque no la totalidad.

Estos resultados deben interpretarse como **asociaciones estadísticas dentro de la muestra analizada**. El trabajo permite comparar mercados y detectar patrones, pero no establecer relaciones causales ni predecir con precisión el precio de cualquier vivienda fuera de las condiciones estudiadas.

Para la memoria del TFM, estas conclusiones deberán vincularse con el análisis exploratorio, los coeficientes beta estandarizados, la significación estadística y las limitaciones del conjunto de datos.


In [ ]:
# =====================================================
# EXPORTACIÓN COMPLEMENTARIA DE DIAGNÓSTICOS
# =====================================================

diagnosticos.to_csv(
    TABLES_DIR / "diagnosticos_modelos.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Tabla de diagnósticos exportada en:", TABLES_DIR)


In [ ]:
# =====================================================
# EXPORTACIÓN DE RESULTADOS FINALES DE VALIDACIÓN Y RANDOM FOREST
# =====================================================

resultados_validacion.to_csv(
    TABLES_DIR / "04_validacion_train_test_modelos.csv",
    index=False,
)

comparacion_test.to_csv(
    TABLES_DIR / "04_comparacion_train_test_modelos.csv",
    index=False,
)

resultados_rf.to_csv(
    TABLES_DIR / "04_resultados_random_forest.csv",
    index=False,
)

comparacion_ols_rf.to_csv(
    TABLES_DIR / "04_comparacion_ols_random_forest.csv",
    index=False,
)

top5_importancias_rf.to_csv(
    TABLES_DIR / "04_top5_importancias_random_forest.csv",
    index=False,
)

print("Resultados finales de validación y Random Forest exportados en:", TABLES_DIR)
